In [21]:
import wandb
import pandas as pd

ENTITY = "laithzumot"
PROJECT = "huggingface" 
RUN_ID = "yqnbvuec"

api = wandb.Api()
run = api.run(f"{ENTITY}/{PROJECT}/runs/{RUN_ID}")


In [12]:
all_steps = list(run.scan_history())
print(f"✅ Retrieved {len(all_steps)} rows from scan_history()")


✅ Retrieved 2000 rows from scan_history()


In [14]:
# Build DataFrame manually to avoid pandas issues
data = []
for step_data in all_steps:
    step = step_data.get('_step')
    reward = step_data.get('train/rewards/lean_reward')
    
    # Only add if we have both values
    if step is not None and reward is not None:
        data.append({'step': int(step), 'reward': float(reward)})
    elif step is not None:
        data.append({'step': int(step), 'reward': None})  # Explicit None for missing rewards

In [15]:
df = pd.DataFrame(data)

In [16]:
print(f"✅ Created DataFrame with {len(df)} rows")
print(f"Step range: {df['step'].min()} to {df['step'].max()}")
print(f"Non-null rewards: {df['reward'].notna().sum()}")

# Show sample before saving
print("\n=== SAMPLE DATA (first 20 rows) ===")
print(df.head(20).to_string(index=False))

# Verify data types
print(f"\nData types: {df.dtypes.to_dict()}")

# Save with explicit formatting
csv_path = '/home/lyz/repos/open-r1-lean/past_runs/run_3_algebra/step_rewards_clean.csv'
df.to_csv(csv_path, index=False, float_format='%.6f')

print(f"\n✅ SAVED TO: {csv_path}")

# Verify the file was written correctly
print("\n=== VERIFYING FILE ===")
loaded_df = pd.read_csv(csv_path)
print(f"Loaded {len(loaded_df)} rows from file")
print(loaded_df.head(10).to_string(index=False))

✅ Created DataFrame with 2000 rows
Step range: 0 to 1637
Non-null rewards: 22

=== SAMPLE DATA (first 20 rows) ===
 step  reward
    0     NaN
    1     NaN
    4     NaN
    7     NaN
    8     NaN
    9     NaN
   11     NaN
   12     NaN
   14     NaN
   16     NaN
   17     NaN
   27     NaN
   28     NaN
   30     NaN
   31     NaN
   32     NaN
   33     NaN
   36     NaN
   38     NaN
   40     NaN

Data types: {'step': dtype('int64'), 'reward': dtype('float64')}

✅ SAVED TO: /home/lyz/repos/open-r1-lean/past_runs/run_3_algebra/step_rewards_clean.csv

=== VERIFYING FILE ===
Loaded 2000 rows from file
 step  reward
    0     NaN
    1     NaN
    4     NaN
    7     NaN
    8     NaN
    9     NaN
   11     NaN
   12     NaN
   14     NaN
   16     NaN


In [36]:
import wandb
import pandas as pd
import json
import os
import requests
from tqdm import tqdm

# --- CONFIGURE ---
ENTITY = "laithzumot"
PROJECT = "huggingface" 
RUN_ID = "yqnbvuec"
OUTPUT_DIR = "/home/lyz/repos/open-r1-lean/past_runs/run_3_algebra/all_completions_raw"
# -----------------

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Initialize API
api = wandb.Api()
run = api.run(f"{ENTITY}/{PROJECT}/runs/{RUN_ID}")

print("=== PHASE 1: GET API KEY ===")
# Get API key from netrc or environment
import netrc
netrc_entry = netrc.netrc().authenticators("api.wandb.ai")
if netrc_entry:
    API_KEY = netrc_entry[2]
else:
    API_KEY = os.getenv('WANDB_API_KEY')
    
if not API_KEY:
    raise ValueError("No API key found. Set WANDB_API_KEY or login with wandb login")

print(f"✓ API key: {API_KEY[:8]}...")

print("\n=== PHASE 2: SCAN ALL STEPS ===")
# Get all history
all_steps = list(run.scan_history())
print(f"✓ Found {len(all_steps)} steps")

print("\n=== PHASE 3: EXTRACT & DOWNLOAD ARTIFACTS (FIXED) ===")
successful_downloads = 0
failed_downloads = 0

# Create session for downloads
session = requests.Session()
session.headers.update({'Authorization': f'Bearer {API_KEY}'})

for step_data in tqdm(all_steps, desc="Processing steps"):
    step = step_data.get('_step')
    artifact_header = step_data.get('completions')
    
    if not artifact_header:
        continue
    
    try:
        # Parse header data flexibly
        if isinstance(artifact_header, dict):
            header_data = artifact_header
        else:
            header_str = str(artifact_header)
            # Try to extract JSON part
            if '\n' in header_str:
                json_part = header_str.split('\n', 1)[1]
            else:
                json_part = header_str
            header_data = json.loads(json_part)
        
        # KEY FIX: Use '_latest_artifact_path' instead of 'artifact_path'
        artifact_path = header_data.get('_latest_artifact_path', '')
        if not artifact_path or 'wandb-client-artifact://' not in artifact_path:
            continue
        
        # FIX: Extract the artifact ID (it's just a long string, not entity/project/name)
        artifact_id = artifact_path.split('://')[1]
        
        # FIX: Use the run's logged_artifacts() to find the artifact by ID
        # This queries the run's output artifacts directly
        target_artifact = None
        for art in run.logged_artifacts():
            if art.id == artifact_id:
                target_artifact = art
                break
        
        if not target_artifact:
            # Fallback: try to construct reference if the above fails
            # (This shouldn't happen with the logged_artifacts approach)
            try:
                artifact_ref = f"{ENTITY}/{PROJECT}/{artifact_id}"
                target_artifact = api.artifact(artifact_ref)
            except:
                failed_downloads += 1
                continue
        
        # Download the file
        file_path = os.path.join(OUTPUT_DIR, f'completions_step_{step}.json')
        
        # Use the artifact's download method directly
        # This is more reliable than manual URL construction
        try:
            # Get the file from the artifact and save it
            target_artifact.get_path("completions.table.json").download(file_path)
            successful_downloads += 1
        except Exception as e:
            # Fallback to direct API call if download() fails
            try:
                # Construct direct API URL using artifact's entity, project, and name
                artifact_ref_full = f"{target_artifact.entity}/{target_artifact.project}/{target_artifact.name}"
                file_url = f"https://api.wandb.ai/artifacts/{artifact_ref_full}/file/completions.table.json"
                
                response = session.get(file_url, timeout=30)
                if response.status_code == 200:
                    with open(file_path, 'wb') as f:
                        f.write(response.content)
                    successful_downloads += 1
                else:
                    failed_downloads += 1
            except:
                failed_downloads += 1
            
    except Exception as e:
        failed_downloads += 1
        # Show first few errors for debugging
        if failed_downloads <= 3:
            print(f"✗ Step {step} error: {str(e)}")

print(f"\n✓ Downloaded: {successful_downloads} files")
print(f"✗ Failed: {failed_downloads} files")

print("\n=== PHASE 4: PROCESS SUCCESSFUL COMPLETIONS ===")
if successful_downloads > 0:
    all_successes = []
    
    for filename in tqdm(os.listdir(OUTPUT_DIR), desc="Processing files"):
        if not filename.startswith('completions_step_'): 
            continue
            
        step = int(filename.replace('completions_step_', '').replace('.json', ''))
        file_path = os.path.join(OUTPUT_DIR, filename)
        
        try:
            with open(file_path, 'r') as f:
                table_data = json.load(f)
            
            df = pd.DataFrame(table_data['data'], columns=table_data['columns'])
            
            # Check if 'reward' column exists
            if 'reward' not in df.columns:
                print(f"⚠️  Step {step}: No 'reward' column found. Available: {df.columns.tolist()}")
                continue
                
            successes = df[df['reward'] == 1.0].copy()
            successes['step'] = step
            all_successes.append(successes)
            
        except Exception as e:
            print(f"✗ Error processing {filename}: {e}")
            continue
    
    if all_successes:
        df_final = pd.concat(all_successes, ignore_index=True)
        
        print(f"\n{'='*60}")
        print(f"🎉 FOUND {len(df_final)} SUCCESSFUL COMPLETIONS!")
        print(f"   Across {df_final['step'].nunique()} steps")
        print(f"{'='*60}")
        
        # Save
        csv_path = '/home/lyz/repos/open-r1-lean/past_runs/run_3_algebra/ALL_successful_completions.csv'
        df_final.to_csv(csv_path, index=False)
        
        print(f"\n📄 SAVED TO: {csv_path}")
        
        # Show first successful proof
        if 'completion' in df_final.columns and len(df_final) > 0:
            first_proof = df_final.iloc[0]
            print(f"\n📝 FIRST SUCCESSFUL PROOF (step {first_proof['step']}):")
            print(f"   {first_proof['completion'][:400]}...")
        
        # Statistics
        print(f"\n📊 STATISTICS:")
        print(f"   Total successful proofs: {len(df_final)}")
        print(f"   Steps with successes: {sorted(df_final['step'].unique())}")
        if successful_downloads > 0:
            avg_per_file = len(df_final) / successful_downloads
            print(f"   Average successes per file: {avg_per_file:.1f}")
        
    else:
        print("❌ No successful completions in downloaded files")
else:
    print("❌ No files were downloaded successfully")

=== PHASE 1: GET API KEY ===
✓ API key: b7972374...

=== PHASE 2: SCAN ALL STEPS ===
✓ Found 2000 steps

=== PHASE 3: EXTRACT & DOWNLOAD ARTIFACTS (FIXED) ===


Processing steps: 100%|██████████| 2000/2000 [21:46<00:00,  1.53it/s]


✓ Downloaded: 0 files
✗ Failed: 378 files

=== PHASE 4: PROCESS SUCCESSFUL COMPLETIONS ===
❌ No files were downloaded successfully
